# Luck-corrected switching headroom on BBOB with GP and PFN surrogates

## Description

**Question.** On standard noise-free benchmarks (COCO/BBOB), with a GP and with the frozen PFN as surrogate, is there **luck-corrected headroom** for switching acquisition functions within a run? That is: does a non-anticipating switching rule beat the tuned best static arm B\*, beyond what an equal-arms null gets? Issue #8.

**Why.** On the PFN's own prior the gain is mixed. The switching pilot (`runs/switching/pilot_20261021-152818`, not snapshotted) shows privileged headroom over B\* (+37% / +36% / +11% at d = 2 / 4 / 6), but a belief controller that loses to B\* (issue #6). The predecessor found no robust gain at all. Meanwhile the literature reports gains from adapting or combining acquisition functions on BBOB and real benchmarks:
- SAWEI adapts EI's exploration weight on noise-free BBOB (Benjamins et al. 2023, arXiv:2306.04262);
- GP-Hedge (arXiv:1009.5419);
- HEBO's acquisition ensemble (arXiv:2012.03826);
- LMABO (arXiv:2602.07904).

Those studies use GP surrogates on functions the surrogate does not model exactly. This experiment moves our luck-controlled design to that setting. It runs both surrogates on the same tasks, so that a GP-only gain can be told apart from a gain that switching has in general.

**Setup.**
- **Tasks.** BBOB f1–f24 (`ioh`, domain [−5, 5]^d mapped to [0, 1]^d), d ∈ {2, 5}. A task is (function, instance, repetition). Repetitions of one instance differ only in the initial design (its shift is keyed by the task seed).
  - *split 0 (selection)*: instance 1, 2 repetitions → 48 tasks/d.
  - *split 1 (evaluation)*: instances 2–5, 2 repetitions → 192 tasks/d (sized by cost: a timing probe on 24 tasks at d = 5 took 274 s (GP) and 173 s (PFN) per privileged member, so the confirm runs fit in about 3 h of ulysses time).

  Nothing is selected on split 1. f is reported as f − f_opt, so f\* = 0 exactly. Regret is log10(regret + 1e-8); 1e-8 is COCO's final target.
- **Budget.** n_init = 2d (scrambled Sobol), T = 10d steps, 5 decisions (every 2d steps). The context is ≤ 60 points: in range for pfn_large (n ≤ 100, d ≤ 6).
- **Surrogates.** Both see y only through its ranks in the current context, a copula transform:
  - **GP**: BoTorch's default single-task GP written out batched. Matérn-5/2 ARD kernel, dimension-scaled LogNormal lengthscale prior, MAP-II refit at every step, on z = Φ⁻¹(rank-uniform y).
  - **PFN**: pfn_large on u = (rank + ½)/n. The marginal of the PFN's training y is Uniform(0, 1) by construction (global ECDF), so u is on the scale it was trained on.

  Neither can use f_opt (ranks are shift-invariant).
- **Arms (K = 6).** ei, pi0.01, ucb0.9, ucb0.99, mes, mean, with the same definitions for both surrogates:
  - the GP uses the closed-form Gaussian versions;
  - the PFN uses the bar distribution.

  ES (alpha-PFN) is left out; its static performance is under question in #6. The candidate set is shared, with CRN per (task, step, replicate): 1024 shifted Sobol points + 3 × 64 incumbent perturbations. Rollouts use 512 + 3 × 32.
- **Methods (per surrogate and d).**
  - `static`: all 6 arms, B\*@0 and random search.
  - `privileged`: the rollout controller. At each decision, R = 8 rollouts per arm on fresh nuisance, each running the arm for one segment and then B\* to the horizon on the true function. It executes the arm with the best mean rollout value. This is one-step policy improvement over B\*, with task knowledge; its gain is unbiased by construction.
  - `equal_arms`: the null control. The same controller over 6 copies of B\*, each restricted to its own random half of the candidates. Its gain over the static B\*@0 must be 0.
- **Not run.** The `belief` controller. Its fantasies live on the per-context rank scale, which has no inverse beyond the observed range, and its value (min fantasised y) is under review in #6. A positive result here is therefore a *ceiling* (switching with task knowledge), not yet a learnable policy.
- **Stages.**
  - `pilot` = selection on split 0 (static only, both surrogates, both d). B\* per (surrogate, d) is the arm with the lowest mean final log10 regret; ties go to the portfolio order. GO if all members are DONE and B\* is defined.
  - `confirm` = evaluation on split 1. One sweep per surrogate, with `task.base_by_d` from the pilot.
- **Numerics.** fp32 (`task.tf32=false`), seed 0, one nuisance stream.

**What to look for.**
- **Primary.** Per surrogate and d: the paired gain of `ctrl:privileged` over static B\* (1 − geometric-mean ratio of final regrets, 95% bootstrap CI over tasks), next to the null's gain over B\*@0.
- **Secondary.**
  1. *How flat is the top:* every static arm vs B\* on split 1 (re-measures the selection), and GP-B\* vs PFN-B\* (which surrogate is better at all).
  2. *Per-function static choice (no switching).* For each function, choose the arm on split 0 (instance 1) and score it on split 1 against the global B\*. This is the value of knowing the problem class, deferred idea 2 and SAWEI's claim. If it is large while switching gains little, the headroom is problem-dependent, not state-dependent.
  3. Gains by BBOB function group (separable 1–5, low/moderate conditioning 6–9, high conditioning 10–14, multimodal with adequate structure 15–19, weak structure 20–24).
  4. Regret curves.
  5. Which arms the controller picks per decision.
  6. Power: MDE at 80% from the per-task sd.

**GO / NO-GO** (per surrogate; `analysis/bo.py::headroom_verdict`, fixed before any run):
- **INVALID** if the null's CI excludes 0 at any d.
- **GO** (luck-corrected headroom detected) if the privileged gain over B\* is ≥ 10% with its CI above 0 at ≥ 1 d, and its point estimate is > 0 at the other d.
- **NO-GO** if the privileged CI upper bound is < 10% at both d. Even with task knowledge, one-step switching then gains less than 10%.
- **INCONCLUSIVE** otherwise.

Reading across surrogates:
- GO for the GP but not for the PFN: the literature's gains come from surrogate misspecification, which a better surrogate removes.
- GO for both: switching has headroom off the prior that our in-prior tests miss.
- NO-GO for both: on noise-free BBOB at this budget, a single tuned arm is as good as one-step switching with task knowledge. Noise (#9) is then the next candidate.

**Limits.**
- Noise-free functions only.
- d ∈ {2, 5}, budget 12d.
- For the PFN, BBOB is off-prior and the rank transform is a modelling choice, not part of the trained model.
- The GP refits from the prior's mode with 100 Adam steps (not L-BFGS); hyperparameters are not shared across steps.
- One-step lookahead over a fixed base: a lower bound on optimal switching, but only with task knowledge (no belief controller).
- One nuisance seed. 24 functions is a small population: per-group results are descriptive only.

In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/bbob_switching/<sweep_id>"  # set after `just snapshot runs/bbob_switching/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.